# Sections 1–6 — preprocessing, stage by stage

This notebook is the **testing** half of the pipeline. Every stage between a raw
C3D file and the fPCA components comes back here as a table, so the shape and
the head of each one can be checked before anything downstream trusts it.

| file | what it holds |
|---|---|
| `sprint_pipeline.py` | sections 1–6, 9, 10 — the functions, data only, no drawing |
| **`sprint_pipeline.ipynb`** | **this notebook — sections 1–6 walked through, `.head()` and `.shape` at every stage** |
| `sprint_outputs.py` | sections 7–8 — every figure and every athlete's MP4, in one call |

Nothing is redefined here. The functions live in `sprint_pipeline.py` and this
notebook calls them in order, so what you test is exactly what the model and the
renderers use.

**The order.** Each section feeds the next, and each one can break the next in a
way that is silent rather than loud:

```
1  settings          paths, capture rate, marker indices
2  anthropometrics   measured height, never derived
3  load → clean      gap-fill, align, cut the lead-in, trim to 62 m
4  segmentation      strides at top speed, steps out of the blocks
5  joint angles      13 angles, size-free by construction
6  fPCA              smooth, decompose, correlate with speed
```

Sections 7 (figures) and 8 (animation) are not here. They render the whole
cohort in one call and there is nothing to inspect a stage at a time — the last
cell runs them.

In [ ]:
from project_paths import PATHS

import numpy as np
import pandas as pd

import sprint_pipeline as SP
from sprint_pipeline import *      # C3D_DIR, clean_for_angles, ANGLE_NAMES, ...

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)


def peek(obj, name="", n=5):
    """The standard check in this notebook: print the shape, show the head.

    Works on a DataFrame, a Series or a plain array — an array is reported by
    shape and dtype only, because a head of a 3-D array is not informative.
    """
    if isinstance(obj, np.ndarray):
        print(f"{name:<26} shape = {obj.shape}   dtype = {obj.dtype}")
        return None
    print(f"{name:<26} shape = {obj.shape}")
    return obj.head(n)


print(f"{len(sorted(C3D_DIR.glob('*.c3d')))} trials found, excluding {EXCLUDED_PIDS}")
print(f"{len(PARTICIPANT_ANTHRO)} participants in the anthropometrics sheet")

## 1 · Paths, capture settings, marker indices

Everything configurable sits at the top of `sprint_pipeline.py` as a module
constant, so no threshold is buried in the middle of a function where it cannot
be found or changed.

Two of these are worth knowing about before reading anything below.

**`ANGLE_STRIDE_MIN_DISTANCE = 18`.** The older code required foot contacts to
be at least 30 frames apart, but a top-speed stride at 60 Hz lasts 27–30 frames.
That filter rejected genuine contacts and glued strides together — see section 4.

**`EXCLUDED_PIDS = ["SB17"]`.** The T8 marker lost tracking on that trial. It is
excluded by name rather than by a quality threshold, because one known bad trial
is a fact about the data, not a rule.

In [ ]:
settings = pd.DataFrame([
    ("SAMPLING_RATE",            SAMPLING_RATE,            "Hz, capture rate"),
    ("TN_POINTS",                TN_POINTS,                "samples per normalised cycle"),
    ("TRIM_DISTANCE_M",          TRIM_DISTANCE_M,          "m, kept from the run"),
    ("SPRINT_DIST_M",            SPRINT_DIST_M,            "m, finish line in the animation"),
    ("SPRINT_START_VEL_THRESH",  SPRINT_START_VEL_THRESH,  "m/s, pelvis forward"),
    ("SPRINT_START_SUSTAIN",     SPRINT_START_SUSTAIN,     "frames it must hold"),
    ("ANGLE_STRIDE_MIN_DISTANCE", ANGLE_STRIDE_MIN_DISTANCE, "frames between contacts"),
    ("STRIDE_PROMINENCE_M",      STRIDE_PROMINENCE_M,      "m, depth of a contact dip"),
    ("STRIDE_FRAMES_MIN",        STRIDE_FRAMES_MIN,        "frames, shortest plausible stride"),
    ("STRIDE_FRAMES_MAX",        STRIDE_FRAMES_MAX,        "frames, longest plausible stride"),
], columns=["setting", "value", "meaning"])

peek(settings, "settings", n=10)

In [ ]:
# The trials themselves. One row per C3D file on disk.
trials_on_disk = pd.DataFrame([
    {"pid": f.stem.split("-")[0].strip(),
     "file": f.name,
     "MB": round(f.stat().st_size / 1e6, 1),
     "excluded": f.stem.split("-")[0].strip() in EXCLUDED_PIDS}
    for f in sorted(C3D_DIR.glob("*.c3d"))
])

print(f"{(~trials_on_disk.excluded).sum()} usable, {trials_on_disk.excluded.sum()} excluded")
peek(trials_on_disk, "trials_on_disk")

## 2 · Anthropometrics — measured, not derived

Height is **measured**, from `60m Participant Anthropometrics.xlsx`. Nothing
derives it from marker positions.

That is not a stylistic preference. A marker-derived height proxy breaks the
moment a marker drops out: it read **2.47 m** for SB17 against a measured
**1.76 m**. There is no reason to estimate a quantity that was taped.

`_load_anthro` raises if the sheet does not hold exactly 31 participants, so a
bad sheet fails at import rather than halfway through an analysis.

The scale factor below is for **drawing only** — averaging athletes of different
sizes would otherwise blur the mean skeleton. The fPCA never sees it: angles are
size-free.

In [ ]:
anthro = pd.DataFrame(PARTICIPANT_ANTHRO).T.sort_index()

print(f"height: {anthro.body_height.min():.2f}–{anthro.body_height.max():.2f} m "
      f"(mean {anthro.body_height.mean():.2f})")
peek(anthro, "anthro (metres)")

In [ ]:
# One scalar per athlete, putting everyone at 1.75 m. Drawing only.
scale = pd.DataFrame({
    "body_height_m": anthro.body_height,
    "scale_to_1.75": [round(height_scale_for(p, 1.75), 3) for p in anthro.index],
})

peek(scale, "scale")

## 3 · Load, align, clean

`clean_for_angles` is the whole of section 3 in one call:

```
load → rough align → cut the lead-in (+settle) → origin reset
     → trim 62 m → refine align
```

Three things in that order matter.

**The alignment runs twice.** A rough pass gives `detect_sprint_start` a forward
axis to work with; the refined pass runs once the standing lead-in is gone.
Refining on clean running only is what squares up the sagittal view.

**Gravity is left alone.** The Xsens IMU supplies a gravity-aligned Z on every
trial, so vertical is correct as recorded. The earlier 3-D PCA re-derived
vertical from variance and rotated the gravity axis into the horizontal plane
whenever lateral spread exceeded height. Only the horizontal plane is rotated
here.

**The block phase is kept.** The start is refined back to the last frame the
hands are still down, so the first steps out of the blocks — the whole point of
the acceleration analysis — survive.

In [ ]:
# Raw, before anything is cleaned.
mk_raw, raw64_raw, fs = load_c3d(find("SB25"))

peek(mk_raw,    "mk (23 segments)")
peek(raw64_raw, "raw64 (64 markers)")
print(f"{'sampling rate':<26} {fs} Hz")

In [ ]:
t = clean_for_angles("SB25")

peek(t["mk"],    "mk, cleaned")
peek(t["raw64"], "raw64, cleaned")
pd.Series({k: v for k, v in t.items() if k not in ("mk", "raw64")}).to_frame("value")

In [ ]:
# Frame by frame, so the cleaning can be read rather than trusted: where the
# pelvis is, where T8 is, and how fast the thorax is going.
vel, peak_f, peak_v = compute_velocity(t["mk"], t["fs"], raw64=t["raw64"])

trace = pd.DataFrame({
    "t_s":        np.arange(t["n_frames"]) / t["fs"],
    "pelvis_x_m": t["mk"][:, IDX_PELVIS, 0],
    "pelvis_z_m": t["mk"][:, IDX_PELVIS, 2],
    "T8_x_m":     t["mk"][:, IDX_T8, 0],
    "r_foot_z_m": t["mk"][:, IDX_R_FOOT, 2],
    "l_foot_z_m": t["mk"][:, IDX_L_FOOT, 2],
    "speed_ms":   vel,
}).round(3)

print(f"frame 0 is the set position; peak speed {peak_v:.2f} m/s at frame {peak_f}")
peek(trace, "trace")

In [ ]:
# The whole cohort, cleaned once and kept. Everything below reuses this, so the
# 31 files are read exactly once.  (~1 minute)
trials = {}
for fpath in sorted(C3D_DIR.glob("*.c3d")):
    pid = fpath.stem.split("-")[0].strip()
    if pid in EXCLUDED_PIDS:
        continue
    try:
        trials[pid] = clean_for_angles(fpath)
    except Exception as e:
        print(f"  skipped {pid}: {e}")

cleaned = pd.DataFrame([
    {"pid": p, "n_frames": tr["n_frames"], "seconds": round(tr["n_frames"] / tr["fs"], 2),
     "tilt_deg": round(abs(tr["tilt_deg"]), 2), "peak_frame": tr["peak_frame"],
     "peak_vel_ms": round(tr["peak_vel_ms"], 2)}
    for p, tr in trials.items()
]).set_index("pid").sort_values("peak_vel_ms", ascending=False)

peek(cleaned, "cleaned")

`tilt_deg` is how far off square the trial was **before** the fix — the angle
between the old SVD-derived forward axis and the athlete's own pelvis
displacement. A large value there means that trial's exported sagittal view was
visibly rotated. `SP.alignment_report()` returns the same column on its own,
re-reading the files; the table above already has it.

In [ ]:
tilt = cleaned[["tilt_deg", "peak_vel_ms"]].sort_values("tilt_deg", ascending=False)

print(f"tilt correction: median {tilt.tilt_deg.median():.2f} deg, "
      f"worst {tilt.tilt_deg.max():.2f} deg")
peek(tilt, "tilt")

## 4 · Segmentation — strides at top speed, steps from the blocks

Before any fPCA the strides have to actually **be** strides.

**Top speed uses same-foot strides.** Contact to contact of the *same* foot is
one full cycle, and at top speed every cycle repeats the last, so they can be
averaged. The older detector required contacts to be ≥ 30 frames apart while a
top-speed stride lasts 27–30, so it rejected real contacts and returned windows
spanning two or three strides glued together. Those windows, time-normalised as
if they were one cycle, are what flattened the averaged curves. The corrected
detector separates contacts by 18 frames and then **validates every window**,
discarding anything outside 22–40 frames instead of silently averaging it.

**Acceleration uses alternate-foot steps.** Out of the blocks the two legs are
doing different jobs, so a same-foot stride would hide the very asymmetry of
interest.

In [ ]:
strides, rejected = find_top_speed_strides(t["mk"], t["peak_frame"])
steps, _          = find_first_steps(t["mk"], n_steps=3)

windows = pd.DataFrame(
    [{"kind": "stride", "n": i + 1, "start": a, "end": b, "frames": b - a,
      "seconds": round((b - a) / t["fs"], 3)} for i, (a, b) in enumerate(strides)] +
    [{"kind": "step",   "n": i + 1, "start": a, "end": b, "frames": b - a,
      "seconds": round((b - a) / t["fs"], 3)} for i, (a, b) in enumerate(steps)]
)

print(f"SB25: {len(strides)} strides kept, {rejected} rejected as implausible")
print(f"      (the old detector returned [54, 30, 32] frames for this trial)")
peek(windows, "windows", n=8)

In [ ]:
# Across the cohort — one row per athlete, so a bad segmentation shows up as an
# outlier rather than as a quietly flattened curve later on.
seg = []
for pid, tr in trials.items():
    st, rej = find_top_speed_strides(tr["mk"], tr["peak_frame"])
    sp, _   = find_first_steps(tr["mk"], n_steps=3)
    seg.append({"pid": pid, "n_strides": len(st), "rejected": rej,
                "mean_stride_frames": round(np.mean([b - a for a, b in st]), 1) if st else np.nan,
                "n_steps": len(sp)})

segmentation = pd.DataFrame(seg).set_index("pid")

print(f"athletes with the full 5 strides : {(segmentation.n_strides == 5).sum()}/{len(segmentation)}")
print(f"athletes with 3 clean steps      : {(segmentation.n_steps == 3).sum()}/{len(segmentation)}")
peek(segmentation, "segmentation")

## 5 · Joint angles

`mk` holds segment **centroids**, not joint centres — "Right Upper Leg" is the
midpoint of the trochanter and the knee, which is neither joint. So angles come
from `raw64`, which carries the anatomical landmarks directly.

**Thirteen angles**: legs, arms and trunk. The arms are in the *analysis*, not
just the drawing — the prior work this project cites reports contralateral
arm–leg timing as what separates faster from slower sprinters.

Each angle is the turn from the **parent segment to the child**. Measuring each
segment against vertical and subtracting looks equivalent but is not: an arm
swinging above the shoulder crosses the ±180° wrap point and the subtraction
then reports a near-full rotation — it gave a 219° shoulder range and a 330° arm
separation, neither of which a person can do. The angle *between* two segments
cannot cross that boundary.

An angle is a ratio of two lengths, so **none of these carry units of body
size**. That is the whole reason the fPCA runs on them.

In [ ]:
angles, names = compute_joint_angles(t["raw64"])
angles_df = pd.DataFrame(angles, columns=names).round(2)

peek(angles_df, "angles_df (SB25)")

In [ ]:
# Range check. Anything here that a human cannot do is a wrap-around bug, not
# an athlete.
angle_ranges = pd.DataFrame({
    "min":  angles.min(0).round(1),
    "max":  angles.max(0).round(1),
    "ROM":  np.ptp(angles, axis=0).round(1),
    "mean": angles.mean(0).round(1),
}, index=names)

peek(angle_ranges, "angle_ranges", n=13)

## 5b · One cycle

0 % of a cycle is foot contact, 100 % is the next contact of the **same** foot,
101 samples. Phase is the only fair axis: stride frequency drifts while an
athlete accelerates, so comparing at equal frame number or equal metres would
compare different parts of the movement.

`stride_cycle_angles` also returns a **phase spread** — how far the moment of
peak knee flexion moves between cycles. If they are the same movement it lands
in nearly the same place every time and averaging is safe; if it scatters, the
average smears the peaks away. That is the guard against the section 4 failure
recurring silently.

In [ ]:
cycle, n_used, spread = stride_cycle_angles(t["raw64"], t["mk"], t["peak_frame"])
cycle_df = pd.DataFrame(cycle, columns=ANGLE_NAMES,
                        index=pd.Index(range(len(cycle)), name="% of cycle")).round(2)

print(f"{n_used} strides averaged, phase spread {spread:.0f} % of the cycle")
peek(cycle_df, "cycle_df")

## 6 · Functional PCA on the angle curves

Both phases are built in one pass: the top-speed cycle from the strides around
peak velocity, and the acceleration cycle from the first three steps, normalised
exactly the same way.

The two **phase spreads** say something important. At top speed every stride
repeats the last, so the spread is small and averaging is safe. The first three
steps are a *progression* — step 1 is not step 3 — so the spread is large by
nature. That average summarises early acceleration; it is not a canonical cycle
the way the top-speed one is.

In [ ]:
cohort, accel = {}, {}
peak_vel, body_height, spread_top, spread_acc = {}, {}, [], []

for pid, tr in trials.items():
    cy, _n, sp = stride_cycle_angles(tr["raw64"], tr["mk"], tr["peak_frame"])
    if cy is None:
        continue
    cohort[pid] = cy
    spread_top.append(sp)

    st, _ = find_first_steps(tr["mk"], n_steps=3)
    if len(st) == 3:
        ac, _na, sa = stride_cycle_angles(tr["raw64"], tr["mk"], tr["peak_frame"],
                                          windows=st)
        if ac is not None:
            accel[pid] = ac
            spread_acc.append(sa)

    peak_vel[pid]    = tr["peak_vel_ms"]
    body_height[pid] = PARTICIPANT_ANTHRO[pid]["body_height"]

# The MEDIAN, not the mean: a couple of athletes with one scattered stride drag
# the mean up and say nothing about the cohort.
print(f"top speed    : {len(cohort)} athletes, phase spread median {np.median(spread_top):.0f} % "
      f"(mean {np.mean(spread_top):.0f} %)")
print(f"acceleration : {len(accel)} athletes, phase spread median {np.median(spread_acc):.0f} % "
      f"(mean {np.mean(spread_acc):.0f} %)")

cohort_df = pd.DataFrame({"peak_vel_ms": pd.Series(peak_vel).round(2),
                          "body_height_m": pd.Series(body_height),
                          "has_accel": [p in accel for p in sorted(peak_vel)]},
                         index=sorted(peak_vel))
peek(cohort_df, "cohort_df")

### Smoothing

A **B-spline** basis, not Fourier. Fourier assumes a fixed repeating frequency,
which is exactly the assumption that fails here: stride frequency drifts and
joint angles have no clean periodic waveform.

Smoothness comes from a **roughness penalty** rather than from hand-picking a
basis size. The **effective degrees of freedom** printed below is the honest
measure of how much wiggle survived — near 15 means barely smoothed, near 2
means almost a straight line.

In [ ]:
X_top = np.stack([cohort[p] for p in sorted(cohort)])
X_acc = np.stack([accel[p]  for p in sorted(accel)])
peek(X_top, "X_top (pid, %, angle)")
peek(X_acc, "X_acc (pid, %, angle)")

smooth_top = dict(zip(sorted(cohort), smooth_angle_curves(X_top, penalty=1.0, report=True)))
smooth_acc = dict(zip(sorted(accel),  smooth_angle_curves(X_acc, penalty=1.0)))

# How much did the smoothing actually change? Per angle, across the cohort.
delta = pd.DataFrame({
    "raw_ROM_deg":      [np.mean([np.ptp(cohort[p][:, j]) for p in sorted(cohort)]) for j in range(13)],
    "smoothed_ROM_deg": [np.mean([np.ptp(smooth_top[p][:, j]) for p in sorted(cohort)]) for j in range(13)],
}, index=ANGLE_NAMES).round(2)
delta["removed_deg"] = (delta.raw_ROM_deg - delta.smoothed_ROM_deg).round(2)

peek(delta, "delta", n=13)

### The decomposition

One decision worth stating plainly: **each angle is standardised to unit
variance before the curves are joined.** Without it PC1 becomes "whichever angle
swings furthest" — knee flexion moves through ~90° while trunk lean moves through
~10°, so the knee would dominate by range alone. That is the same kind of
artefact as measuring body size, wearing a different hat.

There is an automatic check too: if PC1 still explains more than 80 % of the
variance the run prints a loud warning, because that almost always means a size
or scaling artefact survived.

**Velocity must be the unscaled value.** Scaling a trial to a common stature
multiplies every length in it and the velocity derived from it — an athlete
scaled from 1.93 m to 1.75 m has their speed understated by 9 %.

In [ ]:
fpca_top = run_angle_fpca(smooth_top, n_components=6)
fpca_acc = run_angle_fpca(smooth_acc, n_components=6)

scores = pd.DataFrame(fpca_top["scores"],
                      columns=[f"PC{k+1}" for k in range(fpca_top["scores"].shape[1])],
                      index=fpca_top["pids"]).round(3)
scores.insert(0, "peak_vel_ms", [round(peak_vel[p], 2) for p in fpca_top["pids"]])

peek(fpca_top["loadings"], "loadings (PC, %, angle)")
peek(scores, "scores")

In [ ]:
pv = lambda f: np.array([peak_vel[p]    for p in f["pids"]])
bh = lambda f: np.array([body_height[p] for p in f["pids"]])

print(f"TOP SPEED    PC1 = {fpca_top['explained_var'][0]:.1f} % "
      f"(the position-based version was 96.5 %)")
peek(correlate_with_velocity(fpca_top, pv(fpca_top), bh(fpca_top)), "corr, top speed", n=6)

In [ ]:
print(f"ACCELERATION PC1 = {fpca_acc['explained_var'][0]:.1f} %")
peek(correlate_with_velocity(fpca_acc, pv(fpca_acc), bh(fpca_acc)), "corr, acceleration", n=6)

### The PC1 loading vector, segment by segment

A component is not one number per joint — it is a **curve** per joint, 101
points long, 13 of them side by side. So "the loading of the knee" has to be
summarised, and the three summaries below answer different questions:

* **`loading_L2`** — the size of that angle's loading curve over the whole
  cycle. This is the bar in `pc1_loading_ranking.png`. It says how much the
  component *moves* this angle.
* **`share_%`** — `loading_L2²` as a percentage. The loading vector is unit
  length, so these sum to 100 and read directly as "this angle is x % of the
  component".
* **`peak_loading` / `at_%_of_cycle`** — the single largest value, signed, and
  where in the cycle it falls. This is the one that says *what* the component
  does: a negative peak at 88 % means a high score goes with **less** of that
  angle at 88 % of the stride.

`mean_loading` is signed and often near zero even for a large angle. That is not
a contradiction — a component that flexes a joint early and extends it late has
a big curve that averages away.

In [ ]:
loadings_pc1 = pc_loading_table(fpca_top, pc=1,
                                curves_by_pid=smooth_top, peak_vel=peak_vel)

print(f"PC1 explains {fpca_top['explained_var'][0]:.1f} % of the variance; "
      f"share_% sums to {loadings_pc1['share_%'].sum():.0f}")
peek(loadings_pc1, "loadings_pc1", n=13)

In [ ]:
# The loading vector itself, unsummarised: one column per angle, one row per
# percent of the cycle. This is what the table above is describing.
pc1_curves = pd.DataFrame(fpca_top["loadings"][0], columns=ANGLE_NAMES,
                          index=pd.Index(range(fpca_top["n_points"]),
                                         name="% of cycle")).round(4)

peek(pc1_curves, "pc1_curves")

### How the correlation is worked out

Every `r` in this notebook is an ordinary Pearson correlation across the **30
athletes**. The only thing worth spelling out is what the two columns being
correlated actually are, because one of them is built in three steps.

**Step 1 — the input the fPCA actually sees.** Each athlete's cycle is divided
by that angle's cohort-wide SD, then the cohort mean curve is subtracted:

```
Xc[i] = curve[i] / angle_sd  −  mean_curve
```

The division is why one angle cannot dominate by swinging further than another;
the subtraction is what makes it a *variance* decomposition.

**Step 2 — the score is a dot product.** An athlete's PC1 score is that
athlete's centred, standardised cycle multiplied point by point with the PC1
loading, and summed over everything:

```
score[i] = Σ over phase p, angle a of   Xc[i, p, a] · loading[p, a]
```

**Step 3 — the correlation.** Pearson between that column of 30 scores and the
column of 30 peak velocities:

```
r = Σ(x − x̄)(y − ȳ) / √( Σ(x − x̄)² · Σ(y − ȳ)² )
```

**And the per-segment version.** Because step 2 is a plain sum, it splits
exactly: do the inner sum over `p` only and leave `a` alone, and you get each
angle's own contribution to the score. Those 13 partial scores add back up to
the total, so `r_partial_vel` in the table above is the component's correlation
with speed **decomposed by segment** — not a second analysis, just the same sum
grouped differently.

The cell below does all of it by hand and checks each step against the values
the pipeline produced.

In [ ]:
# ── Step 1 · rebuild the fPCA's own input ────────────────────────────────────
pids_f = fpca_top["pids"]
X_f    = np.stack([smooth_top[p] for p in pids_f])          # (30, 101, 13)
Xc     = X_f / fpca_top["angle_sd"] - fpca_top["mean_curve"]
loading_1 = fpca_top["loadings"][0]                          # (101, 13)

peek(X_f,  "X_f  (athlete, %, angle)")
peek(Xc,   "Xc   standardised+centred")
peek(loading_1, "loading_1 (%, angle)")

# ── Step 2 · the score is a dot product, done by hand ────────────────────────
score_by_hand = np.einsum("ipa,pa->i", Xc, loading_1)
print()
print(f"score matches the pipeline : "
      f"{np.allclose(score_by_hand, fpca_top['scores'][:, 0])}  "
      f"(max diff {np.abs(score_by_hand - fpca_top['scores'][:, 0]).max():.2e})")

# ── The same sum, grouped by angle instead of collapsed ──────────────────────
partial, total = pc_scores_by_angle(fpca_top, smooth_top, pc=1)
print(f"13 partial scores add back to the score : "
      f"{np.allclose(partial.sum(axis=1), fpca_top['scores'][:, 0])}")
peek(partial, "partial (athlete, angle)")

In [ ]:
# ── Step 3 · Pearson, written out, then checked against numpy ────────────────
x = fpca_top["scores"][:, 0]                       # PC1 score, one per athlete
y = np.array([peak_vel[p] for p in pids_f])        # peak velocity, unscaled

dx, dy = x - x.mean(), y - y.mean()
r_by_hand = float((dx * dy).sum() / np.sqrt((dx**2).sum() * (dy**2).sum()))
r_numpy   = float(np.corrcoef(x, y)[0, 1])

print(f"n            = {len(x)} athletes")
print(f"r by hand    = {r_by_hand:+.4f}")
print(f"np.corrcoef  = {r_numpy:+.4f}     match: {np.isclose(r_by_hand, r_numpy)}")
print("(this is the r reported for PC1 by correlate_with_velocity)")

# The same arithmetic, once per angle, on that angle's partial score.
by_segment = pd.DataFrame({
    "angle":         ANGLE_NAMES,
    "share_of_PC1_%": (100 * np.linalg.norm(loading_1, axis=0)**2
                       / (np.linalg.norm(loading_1, axis=0)**2).sum()).round(1),
    "r_partial_vel": [round(float(np.corrcoef(partial[:, j], y)[0, 1]), 3)
                      for j in range(partial.shape[1])],
}).sort_values("share_of_PC1_%", ascending=False).reset_index(drop=True)

peek(by_segment, "by_segment", n=13)

Two things are worth reading off that table.

**PC1 is overwhelmingly trunk lean** — around 77 % of the component by share,
with the two hips next at about 7 % each and everything else below 3 %. So "PC1"
in this cohort is close to "how much the athlete leans", and it should be
described that way rather than as a whole-body pattern.

**The correlations are all weak and mostly negative.** PC1 as a whole sits near
r = −0.24 with peak velocity. That is the same conclusion notebook 03 reaches
from the other direction — the components describe the movement well and predict
speed badly. The dominant way these athletes differ is not the way that makes
them fast.

Note that a segment's *share* of the component and its *correlation* with speed
are independent. Trunk lean carries three quarters of PC1 and correlates −0.18;
the elbows carry almost none of it and correlate about as strongly. A big
loading means "this is what the component is made of", not "this is what
matters for speed".

### What do the fast athletes actually do differently?

fPCA finds the directions of greatest variation, which is **not** the same
question as "what makes someone fast". These two tables ask that directly and do
not depend on the components at all — which makes them a check on whether the
components are telling the truth.

In [ ]:
fvs_table, fast, slow, fast_ids, slow_ids = fast_vs_slow_angles(cohort, peak_vel, n=5)

print(f"fast: {', '.join(fast_ids)}")
print(f"slow: {', '.join(slow_ids)}")
peek(fvs_table, "fast_vs_slow", n=13)

In [ ]:
print("TOP SPEED");      rank_top = rank_angles_by_velocity(smooth_top, peak_vel, top=5)
print("\nACCELERATION"); rank_acc = rank_angles_by_velocity(smooth_acc, peak_vel, top=5)
peek(rank_top, "rank_top", n=13)

## Sections 7–8 · Outputs

Everything above is a stage that can be wrong in a way you would want to catch,
so it is checked one table at a time. The output stage is not like that: it has
no intermediate to inspect, and it should run **everyone**, not a chosen
example. So it lives in `sprint_outputs.py` and is called in one line.

```
figures/                 the cohort-wide PNGs — fPCA anatomy, fast vs slow,
                         model comparison and diagnostics
figures/Block Start/     one block-exit kinogram per athlete, all 30
mp4s/                    the three group animations
mp4s/Trial Runs/         one three-panel video per athlete, all 30
```

Anything produced once **per athlete** gets its own Title Case folder, so a set
of thirty stays together instead of burying the handful of figures that describe
the cohort as a whole.

From a shell, `python sprint_outputs.py` does the same thing, and
`--no-trials` skips the per-athlete videos, which are the slow part (roughly
20 s each).

The MP4s render every captured frame at the capture rate — 60 fps, real time —
with fixed axes and a smoothed camera track. The earlier versions dropped two
frames in three and played the rest at 20 fps, which is what made them choppy.

Three group animations, all top 3 against bottom 3: the stride cycle at top
speed, the blocks through 10 steps, and **the whole run** — that last one holds
the entire 62.5 m in one fixed view, so the gap between the groups is visible as
it opens.

In [ ]:
import sprint_outputs as OUT

print(f"figures → {OUT.FIG_OUT}")
print(f"MP4s    → {OUT.MP4_OUT}")

# Everything, for everyone. Uncomment to run — about 15 minutes for the cohort.
# written = OUT.run_all_outputs()

# Or one piece at a time:
# OUT.run_all_outputs(trials=False)          # figures + the group MP4s only
# OUT.render_trial("SB153")                  # one athlete's video
# OUT.render_representative_full(n=3)        # the whole 62.5 m, fast vs slow

In [ ]:
# What is on disk right now.
existing = pd.DataFrame(
    [{"kind": "figure", "folder": f.parent.name, "name": f.name,
      "MB": round(f.stat().st_size / 1e6, 2)}
     for f in sorted(OUT.FIG_OUT.rglob("*.png"))] +
    [{"kind": "mp4", "folder": f.parent.name, "name": f.name,
      "MB": round(f.stat().st_size / 1e6, 2)}
     for f in sorted(OUT.MP4_OUT.rglob("*.mp4"))])

print(existing.groupby(["kind", "folder"]).MB.agg(["count", "sum"]).round(1).to_string()
      if len(existing) else "nothing rendered yet")
peek(existing, "existing")